In [ ]:
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt
import numpy as np

# 1. Load Fashion-MNIST
(xtrain, ytrain), (xtest, ytest) = tf.keras.datasets.fashion_mnist.load_data()

# 2. Normalize pixels: 0-255 -> 0-1
xtrain = xtrain.astype("float32") / 255.0
xtest = xtest.astype("float32") / 255.0

# 3. Add channel dimension
# (28,28) -> (28,28,1)
xtrain = np.expand_dims(xtrain, axis=-1)
xtest = np.expand_dims(xtest, axis=-1)

print("Training shape:", xtrain.shape)
print("Testing shape :", xtest.shape)

# 4. Create CNN
model = Sequential([
    Conv2D(32, (3,3), activation="relu", input_shape=(28,28,1)),
    MaxPooling2D((2,2)),
    Conv2D(64, (3,3), activation="relu"),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.3),
    Dense(10, activation="softmax")
])

# 5. Compile
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# 6. Train
history = model.fit(
    xtrain,
    ytrain,
    epochs=10,
    batch_size=32,
    validation_split=0.2
)

# 7. Evaluate
test_loss, test_accuracy = model.evaluate(xtest, ytest)

print("\nTest Accuracy:", test_accuracy)

# 8. Predictions
probabilities = model.predict(xtest)
ypred = np.argmax(probabilities, axis=1)

# 9. Classification report
class_names = [
    "T-shirt/top",
    "Trouser",
    "Pullover",
    "Dress",
    "Coat",
    "Sandal",
    "Shirt",
    "Sneaker",
    "Bag",
    "Ankle boot"
]

print("\nClassification Report:")
print(classification_report(ytest, ypred, target_names=class_names))

# 10. Confusion matrix
cm = confusion_matrix(ytest, ypred)

plt.figure(figsize=(10,8))
plt.imshow(cm)
plt.colorbar()

plt.xticks(range(10), class_names, rotation=45, ha="right")
plt.yticks(range(10), class_names)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Fashion-MNIST Confusion Matrix")

for i in range(10):
    for j in range(10):
        plt.text(j, i, cm[i,j], ha="center", va="center")

plt.tight_layout()
plt.show()

# 11. Accuracy/loss curves
plt.figure(figsize=(12,4))

plt.subplot(1,2,1)
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.title("Model Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()

plt.subplot(1,2,2)
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Validation")
plt.title("Model Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()

plt.tight_layout()
plt.show()

# 12. Save Day 6 model
model.save("catalog_classifier_v1.keras")

print("\nSaved: catalog_classifier_v1.keras")
